# 📊 Portafolio de Inversión para el Inversor Joven Conservador

**Curso:** Manejo de Datos  
**Enfoque:** Análisis cuantitativo de un portafolio de largo plazo (10–20 años)

---

## 🎯 Filosofía de inversión: ¿Por qué ser conservador siendo joven?

Existe una idea popular que dice *"eres joven, puedes asumir más riesgo"*. Eso es parcialmente cierto — tienes tiempo para recuperarte de caídas. Sin embargo, un **joven conservador** prioriza la consistencia sobre los rendimientos explosivos, por razones muy racionales:

1. **El interés compuesto favorece la paciencia.** Un rendimiento del 9% anual durante 20 años multiplica tu capital por **5.6x**. No necesitas apuestas agresivas.
2. **Las pérdidas grandes tardan más en recuperarse.** Una caída del 50% requiere un +100% para volver al punto de partida.
3. **La volatilidad afecta el comportamiento.** Los inversores que ven caer su portafolio un 40% suelen vender en el peor momento (pánico).

### 📐 Perfil del inversor que modelamos:
- 🎓 25–35 años, inicio de vida profesional
- 🎯 Horizonte de inversión: 15–20 años
- 💡 Objetivo: Crecimiento patrimonial con mínima intervención (estrategia *buy & hold*)
- 🛡️ Tolerancia al riesgo: Baja-Media — acepta volatilidad moderada, evita activos especulativos

---

## 🏗️ Construcción del Portafolio

### ¿Por qué ETFs y no acciones individuales?

Un **ETF (Exchange-Traded Fund)** es una canasta de activos que cotiza en bolsa como si fuera una sola acción. Para un inversor conservador son ideales porque:
- 📦 **Diversificación inmediata** → SPY contiene las 500 empresas más grandes de EE.UU.
- 💸 **Costos bajos** → comisiones de 0.03%–0.20% vs fondos activos que cobran 1%–2%
- 🔍 **Transparencia** → sabes exactamente qué tienes en todo momento
- 📈 **Track record probado** → décadas de datos históricos

### Los activos seleccionados y su justificación:

| Activo | Ticker | Peso | Clase | Justificación |
|--------|--------|------|-------|---------------|
| SPDR S&P 500 ETF | `SPY` | 40% | Renta Variable EE.UU. | Núcleo del portafolio. Exposición a las 500 empresas más grandes del mundo. Rendimiento histórico ~10% anual |
| Invesco QQQ (NASDAQ 100) | `QQQ` | 20% | Renta Variable Tecnología | Crecimiento en tecnología e innovación. Alto potencial a largo plazo con más volatilidad controlada |
| iShares MSCI World ETF | `URTH` | 15% | Renta Variable Global | Diversificación geográfica. Reduce dependencia de EE.UU. (Europa, Japón, mercados emergentes) |
| iShares Core US Aggregate Bond | `AGG` | 15% | Renta Fija | Bono diversificado de EE.UU. Ancla de estabilidad, bajo riesgo, correlación negativa con acciones |
| SPDR Gold Shares | `GLD` | 10% | Materia Prima | Oro como cobertura contra inflación y crisis. Se comporta bien cuando las acciones caen |

> **Nota metodológica:** Los pesos fueron definidos siguiendo el principio de la **Frontera Eficiente de Markowitz** — maximizar rendimiento esperado para un nivel de riesgo dado. Un portafolio 70% renta variable / 15% renta fija / 10% materias primas es clásico para perfil conservador-moderado.

---
## 🔧 Configuración e Instalación

In [1]:
# Instalar dependencias (solo primera vez)
# keras y tensorflow son necesarios para la sección de redes neuronales
!pip install yfinance plotly ipywidgets keras tensorflow scipy --quiet

# Activar widgets en Jupyter/Colab
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    print("✅ Google Colab detectado — widgets activados")
except ImportError:
    print("✅ Jupyter local detectado")

print("✅ Instalación completa")

✅ Jupyter local detectado
✅ Instalación completa


In [2]:
# ─── Importaciones ────────────────────────────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats as scipy_stats
import ipywidgets as widgets
from ipywidgets import VBox, HBox, Layout
from IPython.display import display, HTML, clear_output
from datetime import datetime, timedelta, date
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas correctamente")

✅ Librerías importadas correctamente


---
## 📥 Sección 1: Descarga de Datos

Define el portafolio, selecciona el período y descarga los precios históricos desde Yahoo Finance.  
Presiona el botón para iniciar los datos quedan guardados en memoria para todas las secciones siguientes.

In [3]:
# ════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL PORTAFOLIO (constantes globales)
# ════════════════════════════════════════════════════════════

PORTAFOLIO = {
    'SPY':  {'peso': 0.40, 'nombre': 'S&P 500 ETF',         'color': '#2196F3', 'clase': 'Renta Variable'},
    'QQQ':  {'peso': 0.20, 'nombre': 'NASDAQ 100 ETF',       'color': '#FF5722', 'clase': 'Renta Variable'},
    'URTH': {'peso': 0.15, 'nombre': 'MSCI World ETF',       'color': '#4CAF50', 'clase': 'Renta Variable'},
    'AGG':  {'peso': 0.15, 'nombre': 'US Bonds ETF',         'color': '#9C27B0', 'clase': 'Renta Fija'},
    'GLD':  {'peso': 0.10, 'nombre': 'Gold ETF',             'color': '#FFC107', 'clase': 'Materia Prima'},
}

TICKERS      = list(PORTAFOLIO.keys())
PESOS        = np.array([PORTAFOLIO[t]['peso'] for t in TICKERS])
DIAS_TRADING = 252    # Días hábiles bursátiles en un año
TASA_RF      = 0.045  # Tasa libre de riesgo (~T-Bills 2024)

# Variables globales — se llenarán al presionar el botón
PRECIOS          = None   # DataFrame de precios de cierre por activo
RENDIMIENTOS     = None   # Rendimientos logarítmicos diarios
REND_PORTAFOLIO  = None   # Rendimiento diario del portafolio ponderado

print("📋 Portafolio configurado:")
for t, info in PORTAFOLIO.items():
    print(f"   {t}: {info['nombre']} — {int(info['peso']*100)}%")

📋 Portafolio configurado:
   SPY: S&P 500 ETF — 40%
   QQQ: NASDAQ 100 ETF — 20%
   URTH: MSCI World ETF — 15%
   AGG: US Bonds ETF — 15%
   GLD: Gold ETF — 10%


In [4]:
# ─── Widgets de configuración ─────────────────────────────────────────────────

# Selector de fecha de inicio
start_date_picker = widgets.DatePicker(
    description='Fecha de Inicio:',
    value=date(2005, 1, 1),
    disabled=False,
    style={'description_width': 'initial'}
)

# Selector de fecha de fin
end_date_picker = widgets.DatePicker(
    description='Fecha de Fin:',
    value=date.today(),
    disabled=False,
    style={'description_width': 'initial'}
)

# Output donde se mostrará el resultado de la descarga
descarga_output = widgets.Output()


def descargar_datos(b=None):
    """
    Descarga los precios históricos de Yahoo Finance para los 5 ETFs del portafolio.
    Calcula rendimientos logarítmicos y el rendimiento ponderado del portafolio.
    Guarda los resultados en variables globales para uso en secciones posteriores.
    """
    global PRECIOS, RENDIMIENTOS, REND_PORTAFOLIO

    with descarga_output:
        descarga_output.clear_output()

        print(f"⏳ Descargando datos para: {TICKERS}")
        print(f"   Período: {start_date_picker.value} → {end_date_picker.value}\n")

        try:
            raw = yf.download(
                tickers     = TICKERS,
                start       = str(start_date_picker.value),
                end         = str(end_date_picker.value),
                auto_adjust = True,
                progress    = False
            )

            # yfinance puede devolver MultiIndex — extraemos solo 'Close'
            if isinstance(raw.columns, pd.MultiIndex):
                PRECIOS = raw['Close'][TICKERS].ffill().dropna()
            else:
                PRECIOS = raw[TICKERS].ffill().dropna()

            # Rendimientos logarítmicos diarios: log(P_t / P_{t-1})
            # Son preferidos en finanzas porque son aditivos en el tiempo
            RENDIMIENTOS = np.log(PRECIOS / PRECIOS.shift(1)).dropna()

            # Rendimiento del portafolio = suma ponderada de rendimientos individuales
            REND_PORTAFOLIO = (RENDIMIENTOS * PESOS).sum(axis=1)

            print(f"✅ Datos descargados correctamente:")
            print(f"   → {len(PRECIOS):,} días de trading")
            print(f"   → Período: {PRECIOS.index[0].date()} → {PRECIOS.index[-1].date()}")
            print(f"   → Activos: {list(PRECIOS.columns)}")

            # Tabla de métricas iniciales
            metricas = pd.DataFrame(index=TICKERS)
            metricas['Nombre']           = [PORTAFOLIO[t]['nombre'] for t in TICKERS]
            metricas['Peso (%)']         = (PESOS * 100).round(1)
            metricas['Rend. Anual (%)']  = (RENDIMIENTOS.mean() * DIAS_TRADING * 100).round(2)
            metricas['Volatilidad (%)']  = (RENDIMIENTOS.std() * np.sqrt(DIAS_TRADING) * 100).round(2)
            metricas['Sharpe Ratio']     = (
                (metricas['Rend. Anual (%)']/100 - TASA_RF) /
                (metricas['Volatilidad (%)']/100)
            ).round(3)
            metricas['Rend. Total (%)']  = ((PRECIOS.iloc[-1] / PRECIOS.iloc[0] - 1) * 100).round(2)

            rend_pa  = REND_PORTAFOLIO.mean() * DIAS_TRADING * 100
            vol_pa   = REND_PORTAFOLIO.std()  * np.sqrt(DIAS_TRADING) * 100
            sharpe_p = (rend_pa/100 - TASA_RF) / (vol_pa/100)

            print("\n📊 Métricas del portafolio (período completo):")
            display(metricas)
            print(f"\n🏆 Portafolio combinado:")
            print(f"   Rendimiento anual:  {rend_pa:.2f}%")
            print(f"   Volatilidad anual:  {vol_pa:.2f}%")
            print(f"   Sharpe Ratio:       {sharpe_p:.3f}")

            # Gráfica del valor acumulado del portafolio (base 100)
            valor_port = (1 + REND_PORTAFOLIO).cumprod() * 100

            fig, ax = plt.subplots(figsize=(11, 4))
            for t in TICKERS:
                norm = (PRECIOS[t] / PRECIOS[t].iloc[0]) * 100
                ax.plot(norm.index, norm.values,
                        label=f"{t} — {PORTAFOLIO[t]['nombre']}",
                        color=PORTAFOLIO[t]['color'], linewidth=1.3, alpha=0.75)
            ax.plot(valor_port.index, valor_port.values,
                    label='📊 Portafolio Ponderado',
                    color='black', linewidth=2.5, linestyle='--')
            ax.axhline(100, color='gray', linestyle=':', linewidth=1, alpha=0.6)
            ax.set_title('Rendimiento Acumulado (Base 100) — Período completo', fontsize=13)
            ax.set_xlabel('Fecha')
            ax.set_ylabel('Valor indexado')
            ax.legend(fontsize=8, loc='upper left')
            ax.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()

        except Exception as e:
            print(f"❌ Error al descargar datos: {e}")


# Botón principal de descarga
button_descarga = widgets.Button(
    description='📥 Descargar Datos del Portafolio',
    button_style='primary',
    layout=Layout(width='300px', height='38px')
)
button_descarga.on_click(descargar_datos)

display(HBox([start_date_picker, end_date_picker]))
display(button_descarga, descarga_output)

Button(button_style='primary', description='📥 Descargar Datos del Portafolio', layout=Layout(height='38px', wi…

Output()

---
## 📈 Sección 2: Dashboard Interactivo

Selecciona el período y tipo de visualización, luego presiona el botón para actualizar la gráfica.

In [5]:
# ════════════════════════════════════════════════════════════
# DASHBOARD — Selección de período y tipo de gráfica
# ════════════════════════════════════════════════════════════

# Diccionario de períodos disponibles
PERIODOS = {
    '1 Semana': 7,    '1 Mes': 30,    '3 Meses': 90,
    '6 Meses': 180,   '1 Año': 365,   '3 Años': 365*3,
    '5 Años': 365*5,  '10 Años': 365*10, 'Máximo': 365*20
}

# Widget: selector de período (botones de toggle)
selector_periodo = widgets.ToggleButtons(
    options=list(PERIODOS.keys()),
    value='5 Años',
    style={'button_width': '90px', 'description_width': '0px'},
    layout=Layout(width='100%')
)

# Widget: selector de tipo de gráfica (radio buttons)
selector_tipo = widgets.RadioButtons(
    options=[
        ('Rendimiento acumulado (Base 100)', 'norm'),
        ('Rendimientos diarios (%)',          'rend'),
        ('Volatilidad móvil 30 días',        'vol'),
        ('Drawdown (caída desde máximo)',     'dd'),
    ],
    value='norm',
    style={'description_width': 'initial'},
    layout=Layout(width='370px')
)

# Checkboxes para seleccionar activos
checks_activos = [
    widgets.Checkbox(
        value=True,
        description=f"{t} — {PORTAFOLIO[t]['nombre']} ({int(PORTAFOLIO[t]['peso']*100)}%)",
        style={'description_width': 'initial'},
        layout=Layout(width='400px')
    ) for t in TICKERS
]
check_portafolio = widgets.Checkbox(
    value=True, description='📊 Mostrar portafolio ponderado',
    style={'description_width': 'initial'}, layout=Layout(width='300px')
)

# Output donde aparece la gráfica
dashboard_output = widgets.Output()


def actualizar_dashboard(b=None):
    """
    Genera la gráfica del portafolio según la configuración de los widgets.
    Se activa al presionar el botón 'Actualizar Gráfica'.
    """
    with dashboard_output:
        dashboard_output.clear_output()

        if PRECIOS is None:
            print("❌ Primero descarga los datos con el botón de la Sección 1.")
            return

        activos_sel  = [t for t, cb in zip(TICKERS, checks_activos) if cb.value]
        mostrar_port = check_portafolio.value
        dias_max     = PERIODOS[selector_periodo.value]
        tipo_graf    = selector_tipo.value

        if not activos_sel and not mostrar_port:
            print("⚠️ Selecciona al menos un activo.")
            return

        # Filtrar período seleccionado
        fecha_corte = PRECIOS.index[-1] - timedelta(days=dias_max)
        precios_f   = PRECIOS.loc[PRECIOS.index >= fecha_corte]
        rend_f      = RENDIMIENTOS.loc[RENDIMIENTOS.index >= fecha_corte]
        rend_port_f = REND_PORTAFOLIO.loc[REND_PORTAFOLIO.index >= fecha_corte]

        fig, ax = plt.subplots(figsize=(12, 5))

        if tipo_graf == 'norm':
            titulo, ylabel = f'Rendimiento Acumulado (Base 100) — {selector_periodo.value}', 'Valor indexado'
            if activos_sel:
                norm = (precios_f[activos_sel] / precios_f[activos_sel].iloc[0]) * 100
                for t in activos_sel:
                    ax.plot(norm.index, norm[t].values,
                            label=f"{t} — {PORTAFOLIO[t]['nombre']}",
                            color=PORTAFOLIO[t]['color'], linewidth=2)
            if mostrar_port:
                vport = (1 + rend_port_f).cumprod() * 100
                ax.plot(vport.index, vport.values,
                        label='📊 Portafolio Ponderado',
                        color='black', linewidth=2.8, linestyle='--')
            ax.axhline(100, color='gray', linestyle=':', linewidth=1, alpha=0.6,
                       label='Base inicial (100)')

        elif tipo_graf == 'rend':
            titulo, ylabel = f'Rendimientos Diarios (%) — {selector_periodo.value}', 'Rendimiento (%)'
            for t in activos_sel:
                ax.bar(rend_f.index, rend_f[t].values * 100,
                       label=t, color=PORTAFOLIO[t]['color'], alpha=0.65, width=1)
            if mostrar_port:
                ax.plot(rend_port_f.index, rend_port_f.values * 100,
                        label='Portafolio', color='black', linewidth=1.8)

        elif tipo_graf == 'vol':
            titulo, ylabel = f'Volatilidad Anualizada 30 días (%) — {selector_periodo.value}', 'Volatilidad (%)'
            for t in activos_sel:
                vm = rend_f[t].rolling(30).std() * np.sqrt(252) * 100
                ax.plot(vm.index, vm.values,
                        label=t, color=PORTAFOLIO[t]['color'], linewidth=2)
            if mostrar_port:
                vol_p = rend_port_f.rolling(30).std() * np.sqrt(252) * 100
                ax.plot(vol_p.index, vol_p.values,
                        label='Portafolio', color='black', linewidth=2.5, linestyle='--')

        elif tipo_graf == 'dd':
            titulo, ylabel = f'Drawdown desde Máximo (%) — {selector_periodo.value}', 'Drawdown (%)'
            for t in activos_sel:
                s = precios_f[t]
                dd = (s - s.cummax()) / s.cummax() * 100
                ax.fill_between(dd.index, dd.values, 0,
                                alpha=0.4, color=PORTAFOLIO[t]['color'], label=t)
            if mostrar_port:
                vp = (1 + rend_port_f).cumprod()
                dd_p = (vp - vp.cummax()) / vp.cummax() * 100
                ax.plot(dd_p.index, dd_p.values,
                        label='Portafolio', color='black', linewidth=2.5, linestyle='--')

        ax.set_title(titulo, fontsize=13, fontweight='bold')
        ax.set_xlabel('Fecha')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=8, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Tabla de métricas para el período seleccionado
        if activos_sel and not rend_f[activos_sel].empty:
            m = pd.DataFrame(index=activos_sel)
            m['Nombre']          = [PORTAFOLIO[t]['nombre'] for t in activos_sel]
            m['Rend. Anual (%)'] = (rend_f[activos_sel].mean() * 252 * 100).round(2)
            m['Volatilidad (%)'] = (rend_f[activos_sel].std()  * np.sqrt(252) * 100).round(2)
            m['Sharpe']          = (
                (m['Rend. Anual (%)']/100 - TASA_RF) /
                (m['Volatilidad (%)']/100)
            ).round(3)
            m['Rend. Total (%)'] = (
                (precios_f[activos_sel].iloc[-1]/precios_f[activos_sel].iloc[0] - 1) * 100
            ).round(2)
            print(f"\n📊 Métricas del período seleccionado ({selector_periodo.value}):")
            display(m)


# Botón de actualización
button_dashboard = widgets.Button(
    description='🔄 Actualizar Gráfica',
    button_style='info',
    layout=Layout(width='200px', height='38px')
)
button_dashboard.on_click(actualizar_dashboard)

# Layout del dashboard
panel_checks = VBox(
    [widgets.HTML('<b>📦 Activos</b>')] + checks_activos + [check_portafolio],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='430px')
)
panel_tipo = VBox(
    [widgets.HTML('<b>📉 Visualización</b>'), selector_tipo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px', width='380px')
)
panel_periodo = VBox(
    [widgets.HTML('<b>📅 Período</b>'), selector_periodo],
    layout=Layout(border='1px solid #ddd', padding='10px', border_radius='8px')
)

display(HBox([panel_checks, panel_tipo], layout=Layout(gap='12px', margin='0 0 10px 0')))
display(panel_periodo)
display(button_dashboard)
display(dashboard_output)

Button(button_style='info', description='🔄 Actualizar Gráfica', layout=Layout(height='38px', width='200px'), s…

Output()

---
## 📐 Sección 3: Análisis Cuantitativo — Correlaciones y Simulación histórica

In [6]:
# ─── Análisis cuantitativo: correlación + simulación histórica ────────────────

analisis_output = widgets.Output()

def analisis_cuantitativo(b=None):
    """
    Muestra la matriz de correlación entre activos y simula cuánto
    valdría hoy una inversión inicial de $10,000 hace N años.
    """
    with analisis_output:
        analisis_output.clear_output()

        if PRECIOS is None:
            print("❌ Primero descarga los datos con el botón de la Sección 1.")
            return

        # ── Matriz de correlación ──────────────────────────────
        corr = RENDIMIENTOS.corr().round(3)
        nombres = [PORTAFOLIO[t]['nombre'] for t in TICKERS]

        fig, ax = plt.subplots(figsize=(7, 5))
        im = ax.imshow(corr.values, cmap='RdYlGn', vmin=-1, vmax=1)
        plt.colorbar(im, ax=ax, shrink=0.8)

        ax.set_xticks(range(len(TICKERS)))
        ax.set_yticks(range(len(TICKERS)))
        ax.set_xticklabels(nombres, rotation=30, ha='right', fontsize=9)
        ax.set_yticklabels(nombres, fontsize=9)

        for i in range(len(TICKERS)):
            for j in range(len(TICKERS)):
                ax.text(j, i, f"{corr.values[i,j]:.2f}",
                        ha='center', va='center', fontsize=10, fontweight='bold',
                        color='black' if abs(corr.values[i,j]) < 0.7 else 'white')

        ax.set_title('Correlación entre Rendimientos del Portafolio', fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()
        print("💡 El oro (GLD) y los bonos (AGG) tienen correlación baja/negativa "
              "con acciones → buena diversificación.")

        # ── Simulación histórica: $10,000 invertidos hace 10 años ─────────────
        INVERSION = 10_000
        fecha_10y = PRECIOS.index[-1] - timedelta(days=365*10)
        precios_10y   = PRECIOS.loc[PRECIOS.index >= fecha_10y]
        rend_port_10y = REND_PORTAFOLIO.loc[REND_PORTAFOLIO.index >= fecha_10y]

        valor_port_10y = INVERSION * (1 + rend_port_10y).cumprod()

        fig2, ax2 = plt.subplots(figsize=(11, 5))
        for t in TICKERS:
            valor_t = INVERSION * (precios_10y[t] / precios_10y[t].iloc[0])
            ax2.plot(valor_t.index, valor_t.values,
                     label=f"{t} (100%)", color=PORTAFOLIO[t]['color'],
                     linewidth=1.5, linestyle='--', alpha=0.75)

        ax2.plot(valor_port_10y.index, valor_port_10y.values,
                 label='📊 Portafolio Ponderado', color='black', linewidth=3)
        ax2.axhline(INVERSION, color='gray', linestyle=':', linewidth=1,
                    label=f'Inversión inicial ${INVERSION:,}')

        valor_final = valor_port_10y.iloc[-1]
        ganancia    = valor_final - INVERSION
        mult        = valor_final / INVERSION

        ax2.set_title(f'$10,000 USD invertidos hace 10 años → ${valor_final:,.0f} ({mult:.1f}x)',
                      fontsize=12, fontweight='bold')
        ax2.set_xlabel('Fecha')
        ax2.set_ylabel('Valor del portafolio (USD)')
        ax2.legend(fontsize=8, loc='upper left')
        ax2.grid(True, alpha=0.3)
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        plt.tight_layout()
        plt.show()

        print(f"\n💰 Resultado de la simulación histórica (últimos 10 años):")
        print(f"   Inversión inicial:  ${INVERSION:>10,}")
        print(f"   Valor final:        ${valor_final:>10,.0f}")
        print(f"   Ganancia:           ${ganancia:>10,.0f}")
        print(f"   Múltiplo:           {mult:.2f}x")


button_analisis = widgets.Button(
    description='📊 Mostrar Correlaciones y Simulación Histórica',
    button_style='success',
    layout=Layout(width='380px', height='38px')
)
button_analisis.on_click(analisis_cuantitativo)
display(button_analisis, analisis_output)

Button(button_style='success', description='📊 Mostrar Correlaciones y Simulación Histórica', layout=Layout(hei…

Output()

---
## 🎲 Sección 4: Proyección Monte Carlo — SPY (S&P 500)

### ¿Qué es Monte Carlo?

La simulación de **Monte Carlo** es una técnica matemática que usa **números aleatorios** para modelar situaciones con incertidumbre. En finanzas la usamos para responder: *¿cómo podría evolucionar el precio de un activo en el futuro?*

**Idea central:** Si sabemos cuál ha sido el rendimiento promedio y la volatilidad histórica de un activo, podemos simular miles de posibles "futuros" respetando esas estadísticas. Al ver el conjunto de todos esos futuros, obtenemos una distribución de probabilidad de los precios.

### El modelo: Movimiento Browniano Geométrico (GBM)

$$S_{t+1} = S_t \cdot \exp\left[(\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t} \cdot Z\right]$$

Donde:
- $\mu$ = rendimiento promedio diario (estimado históricamente)
- $\sigma$ = volatilidad diaria (estimada históricamente)
- $Z \sim \mathcal{N}(0,1)$ = número aleatorio normal estándar
- El término $-\sigma^2/2$ es la **corrección de Jensen** que evita sesgo al tomar logaritmos

Usa los sliders para configurar la simulación y presiona el botón para ejecutarla.

In [7]:
# ════════════════════════════════════════════════════════════
# MONTE CARLO — SPY (activo único)
# ════════════════════════════════════════════════════════════

ACTIVO_MC = 'SPY'   # Activo a proyectar (el más representativo del portafolio)

# Widget: número de simulaciones
N_slider_mc = widgets.IntSlider(
    value=500,
    min=100, max=2000, step=100,
    description='N simulaciones:',
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=Layout(width='420px')
)

# Widget: horizonte en años
horizonte_slider_mc = widgets.IntSlider(
    value=10,
    min=1, max=20, step=1,
    description='Horizonte (años):',
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=Layout(width='420px')
)

mc_output = widgets.Output()


def simular_montecarlo_spy(b=None):
    """
    Ejecuta la simulación Monte Carlo (GBM) para SPY.
    Muestra:
    - Todas las trayectorias simuladas + bandas de confianza
    - Histograma de la distribución de precios finales
    - Tabla de probabilidades
    """
    with mc_output:
        mc_output.clear_output()

        if PRECIOS is None:
            print("❌ Primero descarga los datos con el botón de la Sección 1.")
            return

        np.random.seed(42)   # Semilla: asegura que obtengas los mismos resultados siempre

        N        = N_slider_mc.value
        H_ANIOS  = horizonte_slider_mc.value
        N_DIAS   = H_ANIOS * DIAS_TRADING

        # ── Estimar parámetros con los últimos 5 años ──────────
        ultimos_5y = PRECIOS.index[-1] - timedelta(days=5*365)
        rend_spy   = RENDIMIENTOS.loc[RENDIMIENTOS.index >= ultimos_5y, ACTIVO_MC]

        mu_d    = rend_spy.mean()         # Rendimiento promedio diario (µ)
        sigma_d = rend_spy.std()          # Volatilidad diaria (σ)
        S0      = float(PRECIOS[ACTIVO_MC].iloc[-1])   # Precio actual

        print(f"📊 Parámetros estimados para {ACTIVO_MC} (últimos 5 años):")
        print(f"   Precio actual:             ${S0:.2f}")
        print(f"   Rendimiento diario (µ):    {mu_d:.4%}")
        print(f"   Volatilidad diaria (σ):    {sigma_d:.4%}")
        print(f"   Rendimiento anual (µ·252): {mu_d*252:.2%}")
        print(f"   Volatilidad anual (σ·√252):{sigma_d*np.sqrt(252):.2%}")
        print(f"\n🎲 Simulando {N:,} escenarios para {H_ANIOS} años...")

        # ── Simulación vectorizada ─────────────────────────────
        # Z: matriz de choques aleatorios normales estándar
        # shape: (N_DIAS, N) — cada columna es un escenario independiente
        Z = np.random.standard_normal((N_DIAS, N))

        # Corrección de Jensen: (µ - σ²/2) evita que el valor esperado
        # crezca más de lo correcto al usar la exponencial
        drift  = (mu_d - 0.5 * sigma_d**2)   # Tendencia diaria ajustada
        shocks = sigma_d * Z                   # Fluctuaciones aleatorias

        # Precios simulados: S0 × exp(suma acumulada de rendimientos diarios)
        precios_sim = S0 * np.exp(np.cumsum(drift + shocks, axis=0))
        # Añadimos S0 en el día 0
        precios_sim = np.vstack([np.full(N, S0), precios_sim])

        # ── Percentiles por día (para las bandas) ─────────────
        p5   = np.percentile(precios_sim, 5,  axis=1)
        p25  = np.percentile(precios_sim, 25, axis=1)
        p50  = np.percentile(precios_sim, 50, axis=1)
        p75  = np.percentile(precios_sim, 75, axis=1)
        p95  = np.percentile(precios_sim, 95, axis=1)

        dias = np.arange(N_DIAS + 1)   # Eje X en días (sin fechas para simplificar)

        # ── Gráfica 1: Trayectorias + bandas ──────────────────
        fig, ax = plt.subplots(figsize=(12, 5))

        # Trayectorias individuales (max 200 para no saturar)
        n_plot = min(200, N)
        ax.plot(dias, precios_sim[:, :n_plot],
                color='steelblue', linewidth=0.5, alpha=0.05)

        # Banda de confianza 90%
        ax.fill_between(dias, p5, p95, alpha=0.15, color='steelblue', label='Banda 90%')
        # Banda central 50%
        ax.fill_between(dias, p25, p75, alpha=0.30, color='steelblue', label='Banda 50%')

        # Percentiles clave
        ax.plot(dias, p95, color='green',  linewidth=1.5, linestyle='--',
                label=f'P95 → ${p95[-1]:.0f}')
        ax.plot(dias, p50, color='red',    linewidth=2.5,
                label=f'Mediana → ${p50[-1]:.0f}')
        ax.plot(dias, p5,  color='tomato', linewidth=1.5, linestyle='--',
                label=f'P5 → ${p5[-1]:.0f}')

        # Punto de inicio
        ax.scatter(0, S0, color='black', s=60, zorder=5, label=f'Hoy: ${S0:.2f}')

        ax.set_title(f'🎲 Monte Carlo — {ACTIVO_MC} ({N:,} escenarios, {H_ANIOS} años)',
                     fontsize=13, fontweight='bold')
        ax.set_xlabel(f'Días de trading (1 año = 252 días)')
        ax.set_ylabel(f'Precio de {ACTIVO_MC} (USD)')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.legend(fontsize=9, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Gráfica 2: Distribución de precios finales ─────────
        precios_finales = precios_sim[-1, :]

        fig2, ax2 = plt.subplots(figsize=(9, 4))
        ax2.hist(precios_finales, bins=60, density=True,
                 color='steelblue', alpha=0.7, edgecolor='white')

        colores_vl = {'P5': ('tomato', p5[-1]), 'Mediana': ('red', p50[-1]),
                      'P95': ('green', p95[-1]), 'Hoy': ('black', S0)}
        for lbl, (col, val) in colores_vl.items():
            ax2.axvline(val, color=col, linewidth=2, linestyle='--',
                        label=f'{lbl}: ${val:,.0f}')

        ax2.set_title(f'Distribución de Precios de {ACTIVO_MC} en {H_ANIOS} años',
                      fontsize=12, fontweight='bold')
        ax2.set_xlabel(f'Precio final de {ACTIVO_MC} (USD)')
        ax2.set_ylabel('Densidad de probabilidad')
        ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Tabla de probabilidades ────────────────────────────
        prob_gan   = (precios_finales > S0).mean() * 100
        prob_2x    = (precios_finales > S0 * 2).mean() * 100
        prob_perder = (precios_finales < S0 * 0.5).mean() * 100

        print(f"\n📊 Probabilidades estimadas para {ACTIVO_MC} en {H_ANIOS} años:")
        print(f"   Precio actual:                       ${S0:.2f}")
        print(f"   Precio mediano al final:             ${p50[-1]:.2f}")
        print(f"   Prob. de terminar con más:           {prob_gan:.1f}%")
        print(f"   Prob. de duplicar la inversión:      {prob_2x:.1f}%")
        print(f"   Prob. de perder más del 50%:         {prob_perder:.1f}%")


button_mc = widgets.Button(
    description='🎲 Ejecutar Monte Carlo (SPY)',
    button_style='warning',
    layout=Layout(width='280px', height='38px')
)
button_mc.on_click(simular_montecarlo_spy)

display(VBox([N_slider_mc, horizonte_slider_mc, button_mc, mc_output]))

---
## 🤖 Sección 5: Comparativa — Monte Carlo vs Red Neuronal LSTM

### ¿Por qué comparar dos métodos?

| Característica | Monte Carlo (GBM) | Red Neuronal LSTM |
|----------------|-------------------|-------------------|
| **Tipo de modelo** | Estadístico / Probabilístico | Machine Learning / Determinístico |
| **Supuesto clave** | Rendimientos son ruido normal | El mercado tiene patrones aprendibles |
| **Salida** | Distribución (miles de escenarios) | Una sola trayectoria |
| **Interpretabilidad** | Alta — µ y σ tienen significado claro | Baja — "caja negra" |
| **Captura tendencias** | No | Sí (parcialmente) |

### ¿Qué es una LSTM?

Una **LSTM (Long Short-Term Memory)** es un tipo de red neuronal diseñada para series de tiempo. Tiene "memoria" — puede recordar patrones de hace varios pasos.

> ⚠️ Las LSTM son buenas para ajustar datos pasados, pero **no son bolas de cristal**. A largo plazo acumulan errores porque los mercados tienen aleatoriedad irreducible.

Presiona el botón para entrenar la LSTM y hacer la comparativa. **Puede tardar 1–3 minutos.**

In [8]:
# ════════════════════════════════════════════════════════════
# COMPARATIVA: MONTE CARLO vs LSTM
# ════════════════════════════════════════════════════════════
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Silenciar logs de TensorFlow

import keras
from keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

ACTIVO_LSTM = 'SPY'     # Mismo activo — comparamos en igualdad de condiciones
VENTANA     = 60        # Días de historia que la LSTM "mira" para predecir el siguiente

# Variables globales para guardar el modelo y sus predicciones
modelo_lstm         = None
predicciones_futuras = None
scaler_lstm         = None

comparativa_output = widgets.Output()


def entrenar_y_comparar(b=None):
    """
    Pasos:
    1. Normaliza los precios de SPY al rango [0,1]
    2. Crea secuencias de 60 días (ventana deslizante)
    3. Divide 80%/20% entrenamiento/prueba
    4. Construye y entrena la LSTM (2 capas + Dropout)
    5. Evalúa en el conjunto de prueba (datos nunca vistos)
    6. Proyecta 3 años hacia el futuro con ventana deslizante
    7. Grafica la comparativa Monte Carlo vs LSTM
    """
    global modelo_lstm, predicciones_futuras, scaler_lstm

    with comparativa_output:
        comparativa_output.clear_output()

        if PRECIOS is None:
            print("❌ Primero descarga los datos con el botón de la Sección 1.")
            return

        print(f"⏳ Entrenando LSTM para {ACTIVO_LSTM}... (1–3 minutos en CPU)")

        precios_spy = PRECIOS[[ACTIVO_LSTM]].copy()

        # ── Paso 1: Normalización ──────────────────────────────
        # La LSTM necesita datos en [0,1] para que el entrenamiento sea estable.
        # Sin normalizar, los gradientes son enormes y la red no converge.
        scaler_lstm = MinMaxScaler(feature_range=(0, 1))
        precios_esc = scaler_lstm.fit_transform(precios_spy)

        # ── Paso 2: División entrenamiento / prueba ────────────
        # 80% para entrenar, 20% para evaluar honestamente.
        # NUNCA vemos los datos de prueba antes de evaluar (evita "data leakage")
        split      = int(len(precios_esc) * 0.80)
        train_data = precios_esc[:split]
        test_data  = precios_esc[split:]

        # ── Paso 3: Crear secuencias (ventana deslizante) ──────
        # Cada muestra X = últimos 60 precios → predice el precio del día siguiente y
        def crear_seq(datos, ventana):
            X, y = [], []
            for i in range(ventana, len(datos)):
                X.append(datos[i - ventana:i, 0])
                y.append(datos[i, 0])
            return np.array(X), np.array(y)

        X_train, y_train = crear_seq(train_data, VENTANA)
        X_test,  y_test  = crear_seq(
            np.concatenate([train_data[-VENTANA:], test_data]), VENTANA
        )
        # La LSTM necesita shape (muestras, pasos_tiempo, características)
        X_train = X_train.reshape(-1, VENTANA, 1)
        X_test  = X_test.reshape(-1, VENTANA, 1)

        # ── Paso 4: Construir la LSTM ──────────────────────────
        # Arquitectura:
        #   LSTM(64) → Dropout(20%) → LSTM(32) → Dropout(20%) → Dense(16) → Dense(1)
        #
        # Dropout: apaga aleatoriamente el 20% de conexiones durante el entrenamiento.
        # Esto obliga a la red a NO depender de neuronas específicas → generaliza mejor.
        keras.utils.set_random_seed(42)

        modelo_lstm = models.Sequential([
            layers.LSTM(64, return_sequences=True, input_shape=(VENTANA, 1)),
            layers.Dropout(0.20),
            layers.LSTM(32, return_sequences=False),
            layers.Dropout(0.20),
            layers.Dense(16, activation='relu'),
            layers.Dense(1)
        ], name="LSTM_SPY")

        modelo_lstm.compile(optimizer='adam', loss='mean_squared_error')
        modelo_lstm.summary()

        # ── Paso 5: Entrenamiento ──────────────────────────────
        hist = modelo_lstm.fit(
            X_train, y_train,
            epochs=30, batch_size=32,
            validation_split=0.10,
            verbose=0    # Silencioso para no saturar la salida
        )

        print(f"\n✅ Entrenamiento completo")
        print(f"   Loss final (train): {hist.history['loss'][-1]:.6f}")
        print(f"   Loss final (val):   {hist.history['val_loss'][-1]:.6f}")

        # Curva de aprendizaje
        fig0, ax0 = plt.subplots(figsize=(8, 3))
        ax0.plot(hist.history['loss'],     label='Train loss', color='steelblue')
        ax0.plot(hist.history['val_loss'], label='Val loss',   color='tomato', linestyle='--')
        ax0.set_title('Curva de Aprendizaje de la LSTM', fontsize=11, fontweight='bold')
        ax0.set_xlabel('Época')
        ax0.set_ylabel('MSE')
        ax0.legend()
        ax0.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print("💡 Si la pérdida de validación sube mientras la de entrenamiento baja → sobreajuste (overfitting)")

        # ── Paso 6: Evaluación en conjunto de prueba ───────────
        pred_esc = modelo_lstm.predict(X_test, verbose=0)
        pred_real  = scaler_lstm.inverse_transform(pred_esc)
        y_real     = scaler_lstm.inverse_transform(y_test.reshape(-1, 1))

        rmse = np.sqrt(mean_squared_error(y_real, pred_real))
        mae  = mean_absolute_error(y_real, pred_real)
        mape = np.mean(np.abs((y_real - pred_real) / y_real)) * 100

        print(f"\n📊 Métricas de error — conjunto de PRUEBA (datos nunca vistos):")
        print(f"   RMSE (Error cuadrático medio raíz): ${rmse:.2f}")
        print(f"   MAE  (Error absoluto medio):         ${mae:.2f}")
        print(f"   MAPE (Error porcentual absoluto):    {mape:.2f}%")
        print(f"\n   → La LSTM se equivoca en promedio un {mape:.1f}% del precio real")

        fechas_prueba = precios_spy.index[split:]

        fig1, ax1 = plt.subplots(figsize=(12, 5))
        ax1.plot(precios_spy.index[:split], precios_spy[ACTIVO_LSTM].iloc[:split],
                 color='#9E9E9E', linewidth=1.2, label='Histórico (entrenamiento)')
        ax1.plot(fechas_prueba, y_real.flatten(),
                 color='steelblue', linewidth=2.5, label='Precio Real (prueba)')
        ax1.plot(fechas_prueba, pred_real.flatten(),
                 color='tomato', linewidth=2, linestyle='--', label='Predicción LSTM')
        ax1.axvline(x=precios_spy.index[split], color='green',
                    linestyle='--', linewidth=1.5, label='Inicio de prueba')
        ax1.set_title(f'LSTM {ACTIVO_LSTM}: Predicción vs Precio Real (Conjunto de Prueba)',
                      fontsize=12, fontweight='bold')
        ax1.set_xlabel('Fecha')
        ax1.set_ylabel('Precio (USD)')
        ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax1.legend(fontsize=9)
        ax1.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Paso 7: Proyección futura (3 años) con ventana deslizante ──────────
        # Estrategia: predecimos un día a la vez.
        # Cada nueva predicción se añade a la secuencia y usamos esa para predecir el siguiente.
        # ⚠️ Esto acumula errores — la incertidumbre crece con el horizonte.
        DIAS_FUTURO = 252 * 3   # 3 años

        secuencia = precios_esc[-VENTANA:].reshape(1, VENTANA, 1).copy()
        pred_futuras_esc = []

        for _ in range(DIAS_FUTURO):
            siguiente = modelo_lstm.predict(secuencia, verbose=0)[0, 0]
            pred_futuras_esc.append(siguiente)
            secuencia = np.concatenate(
                [secuencia[:, 1:, :], np.array([[[siguiente]]])], axis=1
            )

        predicciones_futuras = scaler_lstm.inverse_transform(
            np.array(pred_futuras_esc).reshape(-1, 1)
        ).flatten()

        S0_lstm = float(precios_spy[ACTIVO_LSTM].iloc[-1])
        print(f"\n✅ Proyección LSTM completada ({DIAS_FUTURO} días / 3 años):")
        print(f"   Precio actual:      ${S0_lstm:.2f}")
        print(f"   Proyección 1 año:   ${predicciones_futuras[251]:.2f}")
        print(f"   Proyección 2 años:  ${predicciones_futuras[502]:.2f}")
        print(f"   Proyección 3 años:  ${predicciones_futuras[-1]:.2f}")

        # ── COMPARATIVA FINAL: Monte Carlo vs LSTM ─────────────
        np.random.seed(42)
        N_COMP   = 500
        N_D_COMP = 252 * 3

        rend_5y = RENDIMIENTOS.loc[
            RENDIMIENTOS.index >= (PRECIOS.index[-1] - timedelta(days=5*365)), ACTIVO_LSTM
        ]
        mu_c, sig_c = rend_5y.mean(), rend_5y.std()

        Z_comp = np.random.standard_normal((N_D_COMP, N_COMP))
        precios_mc = S0_lstm * np.exp(
            np.cumsum((mu_c - 0.5*sig_c**2) + sig_c*Z_comp, axis=0)
        )
        precios_mc = np.vstack([np.full(N_COMP, S0_lstm), precios_mc])

        p5_c  = np.percentile(precios_mc, 5,  axis=1)
        p50_c = np.percentile(precios_mc, 50, axis=1)
        p95_c = np.percentile(precios_mc, 95, axis=1)
        dias_comp = np.arange(N_D_COMP + 1)

        # Histórico reciente (último año)
        precios_1y = precios_spy.loc[
            precios_spy.index >= (precios_spy.index[-1] - timedelta(days=365))
        ]
        dias_hist_rel = np.linspace(-len(precios_1y), 0, len(precios_1y))

        fig2, ax2 = plt.subplots(figsize=(13, 6))

        # Histórico (último año)
        ax2.plot(dias_hist_rel, precios_1y[ACTIVO_LSTM].values,
                 color='black', linewidth=2.5, label='Histórico (último año)')

        # Monte Carlo
        ax2.fill_between(dias_comp, p5_c, p95_c,
                         alpha=0.12, color='steelblue', label='MC: banda 90%')
        ax2.fill_between(dias_comp,
                         np.percentile(precios_mc, 25, axis=1),
                         np.percentile(precios_mc, 75, axis=1),
                         alpha=0.25, color='steelblue', label='MC: banda 50%')
        ax2.plot(dias_comp, p95_c, color='steelblue', linewidth=1.2,
                 linestyle='--', alpha=0.7)
        ax2.plot(dias_comp, p50_c, color='steelblue', linewidth=2.5,
                 label=f'MC: mediana → ${p50_c[-1]:.0f}')
        ax2.plot(dias_comp, p5_c,  color='steelblue', linewidth=1.2,
                 linestyle='--', alpha=0.7)

        # LSTM
        ax2.plot(np.arange(DIAS_FUTURO), predicciones_futuras,
                 color='tomato', linewidth=3, label=f'LSTM → ${predicciones_futuras[-1]:.0f}')

        # Línea vertical "Hoy"
        ax2.axvline(x=0, color='gray', linestyle='--', linewidth=1.5,
                    label='Hoy')
        ax2.scatter(0, S0_lstm, color='black', s=80, zorder=5)

        ax2.set_title(f'🔬 Comparativa: Monte Carlo vs LSTM — {ACTIVO_LSTM} (3 años)',
                      fontsize=13, fontweight='bold')
        ax2.set_xlabel('Días desde hoy (pasado ← 0 → futuro)')
        ax2.set_ylabel('Precio (USD)')
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax2.legend(fontsize=9, loc='upper left')
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        print("\n" + "="*60)
        print("📌 DIFERENCIAS CLAVE:")
        print("="*60)
        print(f"  Monte Carlo (mediana a 3 años): ${p50_c[-1]:.2f}")
        print(f"  LSTM       (proyección 3 años): ${predicciones_futuras[-1]:.2f}")
        print()
        print("  Monte Carlo: muestra UN RANGO de escenarios con probabilidades.")
        print("  LSTM:        muestra UNA TRAYECTORIA (sin incertidumbre explícita).")
        print()
        print("  Para planificación de largo plazo con control de riesgo:")
        print("  → Monte Carlo es más honesto y útil.")
        print("  Para detectar tendencias de corto plazo:")
        print("  → La LSTM puede complementar el análisis.")


button_lstm = widgets.Button(
    description='🤖 Entrenar LSTM y Comparar con Monte Carlo',
    button_style='danger',
    layout=Layout(width='380px', height='38px')
)
button_lstm.on_click(entrenar_y_comparar)
display(button_lstm, comparativa_output)

E0000 00:00:1777594879.251020     545 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777594879.256892     545 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777594879.271518     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271533     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271535     545 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777594879.271536     545 computation_placer.cc:177] computation placer already registered. Please check linka

Button(button_style='danger', description='🤖 Entrenar LSTM y Comparar con Monte Carlo', layout=Layout(height='…

Output()

---
## 💼 Sección 6: Proyección Monte Carlo — Portafolio Completo

### ¿Por qué proyectar el portafolio y no solo SPY?

Cuando proyectamos los 5 activos **juntos**, capturamos la **correlación entre ellos**. Si SPY cae, AGG (bonos) suele subir — ese efecto reduce el riesgo total más que la suma de los riesgos individuales.

Para respetar las correlaciones usamos la **Descomposición de Cholesky**: factoriza la matriz de covarianza $\Sigma = L \cdot L^T$ para generar rendimientos aleatorios correlacionados.

$$\mathbf{r}_{correlacionado} = L \cdot \mathbf{Z}, \quad \mathbf{Z} \sim \mathcal{N}(0, I)$$

Configura los parámetros y presiona el botón.

In [10]:
# ════════════════════════════════════════════════════════════
# MONTE CARLO — PORTAFOLIO COMPLETO (5 activos correlacionados)
# ════════════════════════════════════════════════════════════

# Widget: número de simulaciones
N_slider_port = widgets.IntSlider(
    value=1000,
    min=100, max=3000, step=100,
    description='N simulaciones:',
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=Layout(width='420px')
)

# Widget: horizonte en años
horizonte_slider_port = widgets.IntSlider(
    value=10,
    min=1, max=20, step=1,
    description='Horizonte (años):',
    style={'description_width': 'initial'},
    continuous_update=False,
    layout=Layout(width='420px')
)

# Widget: inversión inicial
inversion_input = widgets.BoundedIntText(
    value=10000,
    min=100, max=1_000_000, step=500,
    description='Inversión inicial (USD):',
    style={'description_width': 'initial'},
    layout=Layout(width='350px')
)

portafolio_mc_output = widgets.Output()


def simular_portafolio_mc(b=None):
    """
    Simulación Monte Carlo multivariada para el portafolio completo.

    Usa la descomposición de Cholesky para que los rendimientos simulados
    de los 5 activos respeten la estructura de correlación histórica.
    Sin Cholesky, simularíamos activos independientes y subestimaríamos
    el riesgo (o sobreestimaríamos la diversificación).
    """
    with portafolio_mc_output:
        portafolio_mc_output.clear_output()

        if PRECIOS is None:
            print("❌ Primero descarga los datos con el botón de la Sección 1.")
            return

        np.random.seed(42)

        N        = N_slider_port.value
        H_ANIOS  = horizonte_slider_port.value
        N_DIAS   = H_ANIOS * DIAS_TRADING
        INV      = inversion_input.value

        print(f"⏳ Simulando portafolio completo: {N:,} escenarios × {H_ANIOS} años")
        print(f"   5 activos correlacionados, inversión inicial: ${INV:,}")

        # ── Parámetros con los últimos 5 años ─────────────────
        ultimos_5y = PRECIOS.index[-1] - timedelta(days=5*365)
        rend_rec   = RENDIMIENTOS.loc[RENDIMIENTOS.index >= ultimos_5y]

        medias   = rend_rec.mean().values       # Medias diarias de cada activo (5,)
        cov_d    = rend_rec.cov().values         # Matriz de covarianza diaria (5×5)
        varianzas = np.diag(cov_d)               # Varianzas individuales (diagonal)

        print(f"\n📐 Medias de rendimiento diario por activo:")
        for t, mu in zip(TICKERS, medias):
            print(f"   {t}: {mu:.4%}")

        # Descomposición de Cholesky: Σ = L · L'
        # L es triangular inferior. Al multiplicar L por vectores de ruido
        # independiente, obtenemos ruido correlacionado con la misma covarianza
        # que los datos históricos. Esta es la clave de la simulación multivariada.
        L = np.linalg.cholesky(cov_d)

        # ── Simulación ─────────────────────────────────────────
        valores_port = np.zeros((N_DIAS + 1, N))
        valores_port[0, :] = INV   # Todos los escenarios empiezan con INV

        drift_vec = medias - 0.5 * varianzas   # Corrección de Jensen para cada activo

        for sim in range(N):
            # Z: ruido normal independiente — shape (5, N_DIAS)
            Z = np.random.standard_normal((len(TICKERS), N_DIAS))

            # Transformar con Cholesky → ruido correlacionado
            # L @ Z produce rendimientos con la covarianza histórica
            shocks = L @ Z   # shape (5, N_DIAS)

            # Rendimientos diarios de cada activo
            rend_activos = drift_vec.reshape(-1, 1) + shocks   # (5, N_DIAS)

            # Rendimiento del portafolio ponderado cada día
            rend_port_d = PESOS @ rend_activos   # (N_DIAS,)

            # Evolución compuesta del valor del portafolio
            valores_port[1:, sim] = INV * np.cumprod(np.exp(rend_port_d))

        # ── Percentiles por día ────────────────────────────────
        p5_p  = np.percentile(valores_port, 5,  axis=1)
        p25_p = np.percentile(valores_port, 25, axis=1)
        p50_p = np.percentile(valores_port, 50, axis=1)
        p75_p = np.percentile(valores_port, 75, axis=1)
        p95_p = np.percentile(valores_port, 95, axis=1)

        dias_p = np.arange(N_DIAS + 1)

        # ── Gráfica 1: Trayectorias + bandas ──────────────────
        fig, ax = plt.subplots(figsize=(12, 6))

        n_plot = min(150, N)
        ax.plot(dias_p, valores_port[:, :n_plot],
                color='steelblue', linewidth=0.4, alpha=0.06)

        ax.fill_between(dias_p, p5_p, p95_p, alpha=0.12, color='steelblue', label='Banda 90%')
        ax.fill_between(dias_p, p25_p, p75_p, alpha=0.28, color='steelblue', label='Banda 50%')

        ax.plot(dias_p, p95_p, color='green',  linewidth=1.5, linestyle='--',
                label=f'P95 → ${p95_p[-1]:,.0f}')
        ax.plot(dias_p, p75_p, color='#8BC34A', linewidth=1.2, linestyle=':',
                label=f'P75 → ${p75_p[-1]:,.0f}')
        ax.plot(dias_p, p50_p, color='tomato', linewidth=3,
                label=f'Mediana → ${p50_p[-1]:,.0f}')
        ax.plot(dias_p, p25_p, color='#FF9800', linewidth=1.2, linestyle=':',
                label=f'P25 → ${p25_p[-1]:,.0f}')
        ax.plot(dias_p, p5_p,  color='red',    linewidth=1.5, linestyle='--',
                label=f'P5 → ${p5_p[-1]:,.0f}')

        ax.axhline(INV, color='gray', linestyle=':', linewidth=1.5,
                   label=f'Inversión inicial ${INV:,}')

        ax.set_title(
            f'💼 Monte Carlo — Portafolio Conservador ({N:,} escenarios, {H_ANIOS} años)\n'
            f'Inversión inicial: ${INV:,} | Mediana a {H_ANIOS} años: ${p50_p[-1]:,.0f}',
            fontsize=12, fontweight='bold'
        )
        ax.set_xlabel(f'Días de trading (1 año ≈ 252 días)')
        ax.set_ylabel('Valor del portafolio (USD)')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax.legend(fontsize=9, loc='upper left')
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Gráfica 2: Distribución de valores finales ─────────
        vals_finales = valores_port[-1, :]

        fig2, ax2 = plt.subplots(figsize=(9, 4))
        ax2.hist(vals_finales, bins=60, density=True,
                 color='steelblue', alpha=0.7, edgecolor='white')

        for pct, col in [(5,'red'), (25,'#FF9800'), (50,'tomato'), (75,'#8BC34A'), (95,'green')]:
            val = np.percentile(vals_finales, pct)
            ax2.axvline(val, color=col, linewidth=2, linestyle='--',
                        label=f'P{pct}: ${val:,.0f}')

        ax2.axvline(INV, color='gray', linewidth=2, linestyle=':',
                    label=f'Inversión: ${INV:,}')
        ax2.set_title(f'Distribución del Portafolio en {H_ANIOS} años (desde ${INV:,})',
                      fontsize=11, fontweight='bold')
        ax2.set_xlabel('Valor final del portafolio (USD)')
        ax2.set_ylabel('Densidad')
        ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
        ax2.legend(fontsize=8)
        ax2.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # ── Tabla de probabilidades ────────────────────────────
        umbral_2x  = INV * 2
        umbral_3x  = INV * 3
        umbral_perd = INV * 0.75

        prob_gan  = (vals_finales > INV).mean() * 100
        prob_2x   = (vals_finales > umbral_2x).mean() * 100
        prob_3x   = (vals_finales > umbral_3x).mean() * 100
        prob_perd = (vals_finales < umbral_perd).mean() * 100

        print(f"\n📊 Resumen de probabilidades — Portafolio en {H_ANIOS} años:")
        print(f"   (Basado en {N:,} simulaciones con ${INV:,} iniciales)\n")
        print(f"   {'Evento':<50} {'Probabilidad':>12}")
        print(f"   {'-'*62}")
        print(f"   {'Terminar con más dinero del que invertiste':<50} {prob_gan:>11.1f}%")
        print(f"   {f'Duplicar inversión (>${umbral_2x:,})':<50} {prob_2x:>11.1f}%")
        print(f"   {f'Triplicar inversión (>${umbral_3x:,})':<50} {prob_3x:>11.1f}%")
        print(f"   {f'Perder más del 25% (< ${umbral_perd:,})':<50} {prob_perd:>11.1f}%")

        print(f"\n📈 Valores esperados al final del horizonte ({H_ANIOS} años):")
        print(f"   P5  (escenario pesimista extremo): ${p5_p[-1]:>12,.0f}  (x{p5_p[-1]/INV:.2f})")
        print(f"   P25 (escenario pesimista):          ${p25_p[-1]:>12,.0f}  (x{p25_p[-1]/INV:.2f})")
        print(f"   P50 (escenario mediano):            ${p50_p[-1]:>12,.0f}  (x{p50_p[-1]/INV:.2f})")
        print(f"   P75 (escenario optimista):          ${p75_p[-1]:>12,.0f}  (x{p75_p[-1]/INV:.2f})")
        print(f"   P95 (escenario optimista extremo):  ${p95_p[-1]:>12,.0f}  (x{p95_p[-1]/INV:.2f})")


button_port_mc = widgets.Button(
    description='💼 Simular Portafolio Completo',
    button_style='primary',
    layout=Layout(width='300px', height='38px')
)
button_port_mc.on_click(simular_portafolio_mc)

display(VBox([N_slider_port, horizonte_slider_port, inversion_input,
              button_port_mc, portafolio_mc_output]))

---
## ✅ Conclusiones

**¿Por qué este portafolio es adecuado para un joven conservador?**

1. **Diversificación real:** 5 activos con correlaciones bajas entre sí — cuando las acciones caen, los bonos y el oro suelen estabilizarse.
2. **Exposición global:** A través de URTH, el portafolio no depende únicamente del mercado americano.
3. **Costos mínimos:** Todos los ETFs tienen expense ratios menores al 0.20% anual.

---

## 🔬 Conclusiones Metodológicas

### Monte Carlo (recomendado para planificación financiera):
- Genera una **distribución completa de escenarios** con probabilidades asociadas
- Es honesto sobre la incertidumbre — la banda se ensancha con el tiempo
- Ideal para: planificación de largo plazo, análisis de riesgo, decisiones de ahorro

### LSTM (complemento para análisis de corto plazo):
- Captura **tendencias y patrones** recientes en los datos
- Produce una sola trayectoria — no modela incertidumbre explícitamente
- Acumula errores en proyecciones largas
- Ideal para: señales de trading de corto plazo, análisis de momentum

> *Ningún modelo puede predecir el futuro con certeza. Monte Carlo lo admite abiertamente mostrando un rango de posibilidades. Para un inversor de largo plazo, esa honestidad sobre la incertidumbre es más valiosa que la aparente precisión de la red neuronal.*

---
> ⚠️ **Disclaimer:** Este notebook es exclusivamente educativo. No constituye asesoramiento financiero ni de inversión. Los rendimientos pasados no garantizan resultados futuros.